# Dunnhumby GAT — 남은 9시드

L4 1개에서 seed42~48·50·51 × 8모형을 순차 실행합니다. 완료된 seed49는 원본 결과로 재사용합니다. 총72개 새 학습이며 각300epoch, 마지막주 신규상품 test 한 번만 평가합니다.

다른 Colab/서버에서 이 시드들을 이미 실행했다면 중복 실행하지 마세요. 이 패키지는 새 Runpod 실행 또는 이 패키지의 동일 출력 재개에만 사용합니다. GPU 요금 자동종료는 제공하지 않습니다.


## 1. 처음 한 번 설치 — 학습 없음
Python 3.10~3.12의 공식 Runpod PyTorch/JupyterLab 템플릿을 사용하세요. 독립 환경에 PyTorch 2.8.0 CUDA12.8과 필요한 패키지를 설치합니다. 이미 이 패키지로 실행한 Pod 재접속 시 설치 셀은 건너뛰세요.

In [ ]:
from pathlib import Path
import subprocess, json, datetime
PACKAGE = Path('/workspace/runpod_gat')
assert (PACKAGE/'launch.py').is_file(), 'runpod_gat.zip을 /workspace에 압축 해제하세요.'
subprocess.run(['bash', str(PACKAGE/'setup.sh'), '--job', 'dh-gat'], check=True)


## 2. 입력 설치 및 실행 범위
기본값은 남은9시드 전체입니다. 먼저 한 시드만 시작하려면 SEEDS=[42]로 바꿀 수 있습니다. 전체 계획은 여전히10시드이며 다른 시드는 나중에 같은 출력에서 이어 실행합니다.


In [ ]:
from pathlib import Path
import subprocess, json, datetime
PACKAGE = Path('/workspace/runpod_gat')
PYTHON = '/workspace/clv/venv/bin/python'
JOB = 'dh-gat'
SEEDS = [42,43,44,45,46,47,48,50,51]
subprocess.run([PYTHON, str(PACKAGE/'launch.py'), 'unpack', '--bundle', str(PACKAGE/'dh_gat_inputs.zip')], check=True)
BASE = [PYTHON, '-u', str(PACKAGE/'launch.py')]
OPTIONS = ['--job', JOB, '--seeds', *map(str, SEEDS)]
print('실행 범위:', JOB, SEEDS, '각 8모형 × 300epoch')


## 3. 원본·GPU·코드 사전점검 — 학습 없음
파일 해시, 고정 코드, CUDA 희소행렬 역전파를 확인합니다. 실제 전체 전처리·메모리 최고 사용량·학습 속도는 다음 실행에서 확인됩니다.

In [ ]:
subprocess.run(BASE + ['check'] + OPTIONS, check=True)


## 4. 학습 시작 또는 이 Runpod 실행 재개
**같은 모델·시드가 다른 코랩/서버에서 실행 중이거나 완료됐으면 실행하지 마세요.**

아래 셀은 별도 프로세스로 시작해 브라우저를 닫아도 Pod가 켜져 있는 동안 계속 실행합니다. 완료 모형은 재학습·재평가하지 않고, 중단된 학습은 같은 Runpod 출력의 마지막 저장 epoch에서 이어갑니다. 저장 폴더·원본·GPU·패키지·실행도구가 달라지면 자동 재개하지 않습니다.

In [ ]:
log_dir = Path('/workspace/clv/logs')
log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir/(JOB + '_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f') + '.log')
with log_path.open('a') as log:
    process = subprocess.Popen(BASE + ['run'] + OPTIONS + ['--confirmed-unstarted'],
        stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print('프로세스:', process.pid, '| 로그:', log_path)
print('시작 요청입니다. 아래 로그에서 입력 검증과 첫 epoch 완료를 확인하세요.')


## 5. 진행 로그 확인
오류가 나면 출력 폴더를 삭제하지 마세요. 최종평가 시작 후 결과 저장 전에 중단된 경우에는 자동 재평가가 차단됩니다. 원본 결과 복구를 먼저 확인해야 합니다.

In [ ]:
if 'log_path' not in globals():
    candidates = sorted(Path('/workspace/clv/logs').glob(JOB + '_*.log'))
    assert candidates, '로그가 없습니다.'
    log_path = candidates[-1]
subprocess.run(['tail', '-n', '45', str(log_path)], check=True)
if 'process' in globals():
    print('실행 중' if process.poll() is None else f'종료 코드: {process.returncode}')


## 6. 결과 받기와 종료
각 모형 완료 후 `/workspace/clv/exports/dh-gat_results_partial.zip`이 갱신됩니다. 새72개와 기존seed49의8개가 모두 모이면 `dh-gat_results.zip`에 10시드 평균·표본표준편차가 추가됩니다. 표준편차는 신뢰구간이나 유의성검정이 아닙니다.

JupyterLab에서 ZIP을 우클릭 → Download 하세요. 가중치·optimizer·재개 상태는 `/workspace/clv/results/dh-gat`에 따로 보존합니다. Mac 백업은 자동이 아닙니다. **완료 또는 오류로 프로세스가 종료돼도 서버 비용은 계속됩니다.** 결과를 내려받고 영구볼륨을 보존한 뒤 콘솔에서 GPU 서버를 종료하세요. Network Volume 요금은 별도입니다.


In [ ]:
for p in sorted(Path('/workspace/clv/exports').glob('*.zip')):
    print(p, f'{p.stat().st_size/1024**2:.2f} MiB')
